# SQL Exploration — Part I

This notebook is the trial-and-error workspace (see `CONVENTIONS.md`). It:
1. Sets up the 4 hypothetical tables from the prompt in DuckDB with synthetic data.
2. Builds each query piece by piece, showing how the result changes at each step.

The final, clean version of each query (without the intermediate steps) is later copied to `sql/sql_queries.sql`.

## 0. Synthetic data setup

[DuckDB](https://duckdb.org/) is used because it is a SQL engine that runs embedded in Python (no separate database server to install), and it integrates well with `pandas`: the data can be generated into a DataFrame and then inserted directly into a real SQL table.

`np.random.seed(42)` fixes the random number generator's "seed": this way, every time this notebook is run, the **same** synthetic data is generated (reproducibility), instead of different numbers each time.

In [1]:
import duckdb
import numpy as np
import pandas as pd

np.random.seed(42)
con = duckdb.connect(database=":memory:")  # in-memory database, scoped to this session

### The 4 tables, with the exact schema from the prompt

`CREATE TABLE` defines each column's name and data type (`VARCHAR` = text, `INT` = integer, `FLOAT` = decimal, `TIMESTAMP` = date+time, `DATE` = date only, `BOOLEAN` = true/false). There are no rows yet, just the empty structure.

**Detail worth flagging for later:** `deliveries.delivery_person_id` is `VARCHAR`, but `delivery_persons.delivery_person_id` is `INT`. This is a real inconsistency in the prompt (not an error introduced here) — later, when joining these two tables, one of the two types will need a `CAST` so they match. For now, Question 1 does not need it, since it involves no join.

In [2]:
con.execute("""
CREATE TABLE deliveries (
    delivery_id VARCHAR,
    delivery_person_id VARCHAR,
    restaurant_area VARCHAR,
    customer_area VARCHAR,
    delivery_distance_km FLOAT,
    delivery_time_min INT,
    order_placed_at TIMESTAMP,
    weather_condition VARCHAR,
    traffic_condition VARCHAR,
    delivery_rating FLOAT
)
""")

con.execute("""
CREATE TABLE delivery_persons (
    delivery_person_id INT,
    name VARCHAR,
    region VARCHAR,
    hired_date DATE,
    is_active BOOLEAN
)
""")

con.execute("""
CREATE TABLE restaurants (
    restaurant_id VARCHAR,
    area VARCHAR,
    name VARCHAR,
    cuisine_type VARCHAR,
    avg_preparation_time_min FLOAT
)
""")

con.execute("""
CREATE TABLE orders (
    order_id INT,
    delivery_id VARCHAR,
    restaurant_id VARCHAR,
    customer_id VARCHAR,
    order_value FLOAT,
    items_count INT
)
""")

print("4 tables created (empty)")

4 tables created (empty)


### Why the synthetic data was generated this way

**180 days** of deliveries are generated (counting back from today), with ~14 deliveries/day on average, spread across 10 customer areas, 15 couriers, and 20 restaurants. `pd.Timestamp.now()` is used as the reference date (not a fixed date) so that, no matter when this notebook is re-run, "the last 30 days" in the queries always lines up with genuinely recent data in the synthetic dataset.

Edge cases seeded on purpose, with the 5 questions in mind:

- **Question 1** (top areas, last 30 days): the **Riverside** area has a normal delivery time for most of the period, but gets +20 min added *only in the last 30 days* (e.g. recent roadworks). **Old Town** is the opposite: it had a serious problem (+25 min) that has since been resolved, so it only shows up in data *older* than the last 30 days. This is used to confirm that the date filter genuinely changes the ranking — if the date `WHERE` were forgotten, Old Town would appear as the worst area, even though it is no longer a current problem.
- **Question 3** (fastest couriers, 50+ deliveries, active): courier `3` is fast, active, and has ~500+ deliveries. Courier `5` is just as fast but **inactive** — this confirms that the `is_active` filter actually excludes them (without it, they would appear as the fastest). Couriers `14` and `15` are "recently hired" with very few deliveries (<50), confirming that the volume filter also excludes them.
- **Question 5** (increasing trend): courier `7` has an effect that makes their delivery time grow month over month (from ~48 to ~60 min), simulating something gradually getting worse (e.g. a route that deteriorated, fatigue, etc.), while the rest of the couriers stay relatively stable.

Questions 2 and 4 did not need special biases: reasonable variety in weather/traffic/cuisine/order value is enough to answer them with data that makes business sense.

In [3]:
today = pd.Timestamp.now().normalize()
n_days = 180
start_date = today - pd.Timedelta(days=n_days)

areas = ["Downtown", "Uptown", "Riverside", "Midtown", "Eastside",
         "Westside", "Old Town", "Suburbia", "Lakeside", "Hillcrest"]
cuisines = ["Italian", "Fast Food", "Sushi", "Indian", "Mexican", "Vegan", "Pizza", "Chinese"]

# each area's own "base" delivery time (e.g. farther or more congested areas)
area_base_time = {a: np.random.uniform(25, 40) for a in areas}

def area_special_effect(area, order_date):
    """Riverside: recent problem (<=30 days). Old Town: already-resolved problem (>30 days)."""
    days_ago = (today - order_date).days
    effect = 0.0
    if area == "Riverside" and days_ago <= 30:
        effect += 20.0
    if area == "Old Town" and days_ago > 30:
        effect += 25.0
    return effect

courier_ids = list(range(1, 16))
inactive_ids = {5, 12}       # couriers no longer working on the platform
new_hire_ids = {14, 15}      # recently hired, still with few deliveries
fast_ids = {3, 5}            # genuinely fast couriers (one active, one inactive)
trend_courier = 7            # courier with an increasing delivery-time trend

weights = []
for cid in courier_ids:
    if cid in fast_ids:
        weights.append(4.0)   # more volume, to guarantee 50+ deliveries
    elif cid in new_hire_ids:
        weights.append(0.05)  # very little volume, to stay under 50 deliveries
    else:
        weights.append(1.0)
weights = np.array(weights)
weights /= weights.sum()

print("Courier setup ready")

Courier setup ready


In [4]:
records = []
delivery_counter = 1

for day_offset in range(n_days):
    order_date = start_date + pd.Timedelta(days=day_offset)
    days_ago = (today - order_date).days
    n_today = np.random.poisson(14)

    for _ in range(n_today):
        courier_id = int(np.random.choice(courier_ids, p=weights))
        area = np.random.choice(areas)
        restaurant_area = np.random.choice(areas)
        distance = round(float(np.random.uniform(1, 15)), 2)
        weather = np.random.choice(["Clear", "Rainy", "Cloudy", "Stormy"], p=[0.55, 0.2, 0.2, 0.05])
        traffic = np.random.choice(["Low", "Medium", "High"], p=[0.4, 0.4, 0.2])

        base_time = 20 + distance * 1.5
        weather_effect = {"Clear": 0, "Cloudy": 2, "Rainy": 8, "Stormy": 15}[weather]
        traffic_effect = {"Low": 0, "Medium": 6, "High": 14}[traffic]
        area_effect = area_base_time[area] - 30
        special_effect = area_special_effect(area, order_date)
        courier_effect = -8.0 if courier_id in fast_ids else 0.0
        trend_effect = ((n_days - days_ago) / n_days) * 20 if courier_id == trend_courier else 0.0
        noise = np.random.normal(0, 4)

        delivery_time = (base_time + weather_effect + traffic_effect + area_effect
                          + special_effect + courier_effect + trend_effect + noise)
        delivery_time = int(np.clip(delivery_time, 10, 120))

        order_time = order_date + pd.Timedelta(hours=float(np.random.uniform(8, 22)))
        rating = round(float(np.clip(np.random.normal(4.3, 0.5), 1, 5)), 1)

        records.append({
            "delivery_id": f"D{delivery_counter:06d}",
            "delivery_person_id": str(courier_id),  # VARCHAR, matching the deliveries table schema
            "restaurant_area": restaurant_area,
            "customer_area": area,
            "delivery_distance_km": distance,
            "delivery_time_min": delivery_time,
            "order_placed_at": order_time,
            "weather_condition": weather,
            "traffic_condition": traffic,
            "delivery_rating": rating,
        })
        delivery_counter += 1

deliveries_df = pd.DataFrame(records)
print(f"{len(deliveries_df)} deliveries generated")
deliveries_df.head()

2550 deliveries generated


,delivery_id,delivery_person_id,restaurant_area,customer_area,delivery_distance_km,delivery_time_min,order_placed_at,weather_condition,traffic_condition,delivery_rating
0,D000001,3,Downtown,Eastside,5.26,26,2026-02-12 09:57:10.490576862,Clear,Medium,3.5
1,D000002,3,Old Town,Riverside,14.77,52,2026-02-12 08:11:08.554042457,Clear,High,5.0
2,D000003,12,Hillcrest,Uptown,5.26,39,2026-02-12 14:55:56.916269608,Clear,Medium,4.2
3,D000004,1,Midtown,Downtown,3.55,35,2026-02-12 08:26:18.189939760,Cloudy,Medium,3.6
4,D000005,11,Hillcrest,Uptown,13.53,71,2026-02-12 13:26:29.335400349,Rainy,High,4.1


In [5]:
# delivery_persons
regions = ["North", "South", "East", "West"]
dp_records = []
for cid in courier_ids:
    if cid in new_hire_ids:
        hired = today - pd.Timedelta(days=int(np.random.uniform(5, 20)))
    else:
        hired = today - pd.Timedelta(days=int(np.random.uniform(60, 1000)))
    dp_records.append({
        "delivery_person_id": cid,
        "name": f"Courier_{cid:02d}",
        "region": np.random.choice(regions),
        "hired_date": hired.date(),
        "is_active": cid not in inactive_ids,
    })
delivery_persons_df = pd.DataFrame(dp_records)

# restaurants: 2 per area, for area/cuisine variety
r_records = []
rid = 1
for area in areas:
    for _ in range(2):
        r_records.append({
            "restaurant_id": f"R{rid:03d}",
            "area": area,
            "name": f"Restaurant_{rid:03d}",
            "cuisine_type": np.random.choice(cuisines),
            "avg_preparation_time_min": round(float(np.random.uniform(10, 30)), 1),
        })
        rid += 1
restaurants_df = pd.DataFrame(r_records)
restaurants_by_area = restaurants_df.groupby("area")["restaurant_id"].apply(list).to_dict()

# orders: one order per delivery, with a restaurant from the same area as restaurant_area
o_records = []
for i, row in deliveries_df.iterrows():
    restaurant_id = np.random.choice(restaurants_by_area[row["restaurant_area"]])
    items = int(np.random.randint(1, 8))
    order_value = round(items * float(np.random.uniform(8, 20)), 2)
    o_records.append({
        "order_id": i + 1,
        "delivery_id": row["delivery_id"],
        "restaurant_id": restaurant_id,
        "customer_id": f"C{np.random.randint(1, 500):04d}",
        "order_value": order_value,
        "items_count": items,
    })
orders_df = pd.DataFrame(o_records)

print("delivery_persons, restaurants, and orders generated")

delivery_persons, restaurants, and orders generated


In [6]:
delivery_persons_df.head(), restaurants_df.head(), orders_df.head()

(   delivery_person_id        name region  hired_date  is_active
 0                   1  Courier_01  North  2024-04-28       True
 1                   2  Courier_02   West  2025-04-20       True
 2                   3  Courier_03   West  2026-02-14       True
 3                   4  Courier_04  South  2024-06-28       True
 4                   5  Courier_05   East  2026-01-11      False,
   restaurant_id       area            name cuisine_type  \
 0          R001   Downtown  Restaurant_001      Chinese   
 1          R002   Downtown  Restaurant_002        Sushi   
 2          R003     Uptown  Restaurant_003      Chinese   
 3          R004     Uptown  Restaurant_004    Fast Food   
 4          R005  Riverside  Restaurant_005      Mexican   
 
    avg_preparation_time_min  
 0                      12.4  
 1                      20.3  
 2                      26.0  
 3                      23.1  
 4                      15.4  ,
    order_id delivery_id restaurant_id customer_id  order_va

### Inserting the DataFrames into the SQL tables

`INSERT INTO table SELECT * FROM df` copies the rows from the pandas DataFrame into the DuckDB table. DuckDB automatically recognizes Python variables holding a DataFrame (which is why `FROM deliveries_df` can be written directly inside the SQL, with no extra registration step).

In [7]:
con.execute("INSERT INTO deliveries SELECT * FROM deliveries_df")
con.execute("INSERT INTO delivery_persons SELECT * FROM delivery_persons_df")
con.execute("INSERT INTO restaurants SELECT * FROM restaurants_df")
con.execute("INSERT INTO orders SELECT * FROM orders_df")

for t in ["deliveries", "delivery_persons", "restaurants", "orders"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows")

deliveries: 2550 rows
delivery_persons: 15 rows
restaurants: 20 rows
orders: 2550 rows


### Quick check of the edge cases

Before moving on to the queries, this confirms that the synthetic data actually has the properties it was designed to have.

In [8]:
# Riverside (recent problem) vs Old Town (already-resolved problem)
con.sql("""
    SELECT
        customer_area,
        ROUND(AVG(delivery_time_min), 1) AS avg_all_time,
        ROUND(AVG(CASE WHEN order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY
                       THEN delivery_time_min END), 1) AS avg_last_30d
    FROM deliveries
    WHERE customer_area IN ('Riverside', 'Old Town')
    GROUP BY customer_area
""").df()

,customer_area,avg_all_time,avg_last_30d
0,Riverside,45.7,62.2
1,Old Town,55.0,34.2


In [9]:
# Key couriers for Question 3: volume and active/inactive status
con.sql("""
    SELECT
        dp.delivery_person_id,
        dp.is_active,
        COUNT(*) AS n_deliveries,
        ROUND(AVG(d.delivery_time_min), 1) AS avg_time
    FROM deliveries d
    JOIN delivery_persons dp ON d.delivery_person_id = CAST(dp.delivery_person_id AS VARCHAR)
    WHERE dp.delivery_person_id IN (3, 5, 14, 15)
    GROUP BY dp.delivery_person_id, dp.is_active
    ORDER BY dp.delivery_person_id
""").df()

,delivery_person_id,is_active,n_deliveries,avg_time
0,3,True,537,37.1
1,5,False,539,37.6
2,14,True,6,38.0
3,15,True,9,42.7


In [10]:
# Courier 7: increasing trend month over month (for Question 5)
con.sql("""
    SELECT
        DATE_TRUNC('month', order_placed_at) AS month,
        ROUND(AVG(delivery_time_min), 1) AS avg_time,
        COUNT(*) AS n
    FROM deliveries
    WHERE delivery_person_id = '7'
    GROUP BY 1
    ORDER BY 1
""").df()

,month,avg_time,n
0,2026-02-01,47.9,12
1,2026-03-01,52.4,16
2,2026-04-01,55.5,22
3,2026-05-01,59.6,19
4,2026-06-01,58.4,19
5,2026-07-01,57.4,20
6,2026-08-01,59.1,11


Confirmed:
- **Old Town** has the highest historical average (~55 min) but drops sharply in the last 30 days (~34 min) — the problem has already been resolved. **Riverside** is the opposite: a moderate historical average (~46 min) that spikes in the last 30 days (~62 min) — a recent problem. If the date filter were forgotten in Question 1, Old Town would appear as the worst area, hiding Riverside's real, current problem.
- Courier `3` (active) and courier `5` (inactive) both have 500+ deliveries and similarly fast times — without the `is_active` filter, courier `5` would show up in the fastest-couriers ranking even though they no longer work there. Couriers `14` and `15` have fewer than 50 deliveries, so they are correctly left out of the ranking due to insufficient volume.
- Courier `7` goes from ~48 min on average to ~60 min over the 6 months — a clear increasing trend, with some month-to-month noise, as would be expected with real data.

With the data ready, the next step is building the queries.

## Question 1: Top 5 customer areas with the highest average delivery time in the last 30 days

This is built in 3 steps: first the basic `GROUP BY` + `AVG()`, over the whole history; then the date filter is added; and finally `ORDER BY` + `LIMIT` narrows it down to the top 5.

### Step 1: basic `GROUP BY` + `AVG()`

`GROUP BY customer_area` groups all rows in `deliveries` that share the same customer area into a single "bucket" per area. `AVG(delivery_time_min)` then computes, within each bucket, the average of the `delivery_time_min` column. The result is one row per area, with its average delivery time — but **using the entire history**, still with no date filter, and in an arbitrary order (there is no `ORDER BY` yet).

In [11]:
con.sql("""
    SELECT
        customer_area,
        AVG(delivery_time_min) AS avg_delivery_time_min
    FROM deliveries
    GROUP BY customer_area
""").df()

,customer_area,avg_delivery_time_min
0,Westside,32.966292
1,Downtown,37.324219
2,Eastside,33.558140
3,Old Town,55.018519
4,Midtown,39.848249
5,Riverside,45.688000
6,Uptown,45.551724
7,Hillcrest,43.175781
8,Lakeside,41.877049
9,Suburbia,45.017316


This shows the 10 areas with their historical average. Note that **Old Town** shows up with the highest average (~55 min) — but, as confirmed in the check above, that problem has already been resolved and should not count toward "the last 30 days". That is why the next step is needed.

### Step 2: adding the date filter (last 30 days)

`WHERE order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY` filters the rows *before* they are grouped: only deliveries whose order date/time (`order_placed_at`) falls within the last 30 days enter the calculation. `CURRENT_DATE` is today's date (per the clock of the machine running the query), and `- INTERVAL 30 DAY` subtracts 30 days from it. Order matters here: SQL applies the `WHERE` before the `GROUP BY`, so the average is already computed using only the recent rows.

In [12]:
con.sql("""
    SELECT
        customer_area,
        AVG(delivery_time_min) AS avg_delivery_time_min
    FROM deliveries
    WHERE order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY
    GROUP BY customer_area
""").df()

,customer_area,avg_delivery_time_min
0,Midtown,41.535714
1,Riverside,62.238095
2,Uptown,47.055556
3,Westside,33.022727
4,Downtown,38.275000
5,Lakeside,42.022222
6,Hillcrest,42.222222
7,Eastside,32.675000
8,Old Town,34.153846
9,Suburbia,47.307692


The result changed as expected: **Old Town** now has one of the lowest averages (~34 min), and **Riverside** rose to the highest (~62 min). This is exactly the point of the exercise: without the date filter, an area that has already recovered would have been flagged as "the problem", missing the real, current one.

### Step 3: `ORDER BY` + `LIMIT` (top 5)

`ORDER BY avg_delivery_time_min DESC` sorts rows from highest to lowest average (`DESC` = descending; the *highest* times are wanted first, since the goal is to find the areas with the most delay). `LIMIT 5` trims the result down to the first 5 rows — the top 5 the question asks for.

In [13]:
con.sql("""
    SELECT
        customer_area,
        AVG(delivery_time_min) AS avg_delivery_time_min
    FROM deliveries
    WHERE order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY
    GROUP BY customer_area
    ORDER BY avg_delivery_time_min DESC
    LIMIT 5
""").df()

,customer_area,avg_delivery_time_min
0,Riverside,62.238095
1,Suburbia,47.307692
2,Uptown,47.055556
3,Hillcrest,42.222222
4,Lakeside,42.022222


This is the complete query for Question 1: **Riverside, Suburbia, Uptown, Hillcrest, and Lakeside** are the 5 areas with the highest average delivery time in the last 30 days, with Riverside clearly ahead of the rest (~62 min vs. ~42-47 min for the following ones).

**Stopping here** — as agreed, this query gets reviewed before it is copied to `sql_queries.sql` and before moving on to Questions 2-5.